# 프로젝트 2 — Weekend 3: 실전 데이터 기반 추천 시스템

**프로젝트**: 검색형 RAG 기반 금융 상품(ETF) 추천 시스템

**이번 주 목표**:
2. 토큰화 품질 정량 비교, 검색 전략 자동 라우팅
3. MAP@K · LLM-as-Judge로 검색 품질 평가 (기존 Hit Rate/MRR 대신)
4. 콘텐츠 기반 추천 **다양성 확보**
5. 통합 파이프라인 + LLM 리포트

In [10]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken finance-datareader kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 22.1 MB/s eta 0:00:00


In [72]:
# 코랩 출력결과 창 높이 설정
from IPython.display import Javascript, display

def resize_colab_cell():
    display(Javascript('google.colab.output.setIframeHeight(0, true, {maxHeight: 300})'))

get_ipython().events.register('pre_run_cell', resize_colab_cell)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# ▶ 스타터 코드 1/5: 환경 설정 + ETF 데이터
import os, json, time, re, math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import Counter
from dataclasses import dataclass, asdict
from numpy.linalg import norm
#from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS as LangchainFAISS
#from langchain.schema import Document
from langchain_core.documents import Document

#load_dotenv()
# 코랩
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL, temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Day 28에서 FDR로 수집한 ETF 데이터 (정적 스냅샷)
etf_data = [
    {"name": "KODEX 200",             "category": "국내주식", "expense_ratio": 0.15, "return_1y":  8.5, "risk_level": "중간", "dividend_yield": 1.8, "volatility": 15.2,
     "keywords": ["코스피", "대형주", "인덱스", "분산투자", "국내주식"],
     "description": "KOSPI 200 지수를 추종하는 대표 국내 ETF. 대형주 중심 분산투자에 적합합니다."},
    {"name": "KODEX 미국S&P500TR",    "category": "해외주식", "expense_ratio": 0.05, "return_1y": 25.3, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 18.5,
     "keywords": ["S&P500", "미국", "대형주", "패시브", "해외주식"],
     "description": "미국 S&P500 지수의 총수익(TR)을 추종. 저비용으로 미국 대형주 전체에 분산투자합니다."},
    {"name": "ACE 미국배당다우존스",   "category": "배당",     "expense_ratio": 0.01, "return_1y": 12.1, "risk_level": "낮음", "dividend_yield": 3.5, "volatility": 10.3,
     "keywords": ["미국배당", "다우존스", "고배당", "월배당", "안정"],
     "description": "미국 고배당 대형주 중심. 최저 수수료(0.01%)로 안정적 배당수익을 추구합니다."},
    {"name": "TIGER 2차전지테마",     "category": "테마",     "expense_ratio": 0.45, "return_1y":-15.2, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 35.7,
     "keywords": ["2차전지", "배터리", "성장주", "고위험", "테마"],
     "description": "2차전지·배터리 관련 기업에 집중 투자. 고성장 기대 대신 높은 변동성을 감수해야 합니다."},
    {"name": "TIGER 국고채10년",      "category": "채권",     "expense_ratio": 0.07, "return_1y":  4.2, "risk_level": "낮음", "dividend_yield": 2.8, "volatility":  5.1,
     "keywords": ["국고채", "10년", "안전자산", "금리", "채권"],
     "description": "한국 10년 국고채 지수를 추종하는 안전자산 ETF. 금리 하락기에 유리합니다."},
    {"name": "KODEX 골드선물(H)",     "category": "원자재",   "expense_ratio": 0.68, "return_1y": 18.7, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 20.4,
     "keywords": ["금", "골드", "인플레이션", "안전자산", "원자재"],
     "description": "금 선물 가격을 추종하며 환헤지 적용. 인플레이션 방어 및 포트폴리오 분산용입니다."},
    {"name": "KODEX 미국나스닥100TR", "category": "해외주식", "expense_ratio": 0.05, "return_1y": 32.1, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 25.3,
     "keywords": ["나스닥100", "미국", "기술주", "성장", "해외주식"],
     "description": "나스닥100 기술주 중심 총수익 ETF. 높은 성장성과 높은 변동성을 동반합니다."},
    {"name": "TIGER 미국반도체",      "category": "섹터",     "expense_ratio": 0.49, "return_1y": 45.0, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 38.2,
     "keywords": ["반도체", "미국", "필라델피아", "기술", "섹터"],
     "description": "미국 필라델피아 반도체 지수 추종. AI/반도체 수요 수혜 기대되나 변동성 극심합니다."},
    {"name": "KODEX 단기채권PLUS",    "category": "채권",     "expense_ratio": 0.03, "return_1y":  3.8, "risk_level": "낮음", "dividend_yield": 3.2, "volatility":  1.5,
     "keywords": ["단기채", "안전", "저위험", "현금성", "채권"],
     "description": "단기 채권 중심의 초저위험 ETF. 현금성 자산 대용으로 안정적 수익을 제공합니다."},
    {"name": "TIGER 리츠부동산",      "category": "리츠",     "expense_ratio": 0.29, "return_1y":  6.5, "risk_level": "중간", "dividend_yield": 4.8, "volatility": 12.8,
     "keywords": ["리츠", "부동산", "배당", "인프라", "실물자산"],
     "description": "국내 리츠·부동산 인프라에 투자. 높은 배당수익률(4.8%)과 실물자산 분산 효과가 있습니다."},
    {"name": "KODEX 200TR",           "category": "국내주식", "expense_ratio": 0.12, "return_1y":  9.1, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 14.9,
     "keywords": ["코스피200", "TR", "총수익", "인덱스", "국내주식"],
     "description": "KODEX 200의 총수익(TR) 버전. 배당 재투자 효과를 포함하여 장기 성과가 우수합니다."},
    {"name": "TIGER 고배당저변동",    "category": "배당",     "expense_ratio": 0.30, "return_1y":  7.2, "risk_level": "낮음", "dividend_yield": 5.2, "volatility":  8.7,
     "keywords": ["고배당", "저변동", "방어적", "배당", "안정"],
     "description": "고배당+저변동성 종목을 선별. 배당수익률 5.2%로 방어적 포트폴리오에 적합합니다."},
]

print(f"✅ ETF 데이터 {len(etf_data)}개 로드")

✅ ETF 데이터 12개 로드


In [12]:
# ▶ 스타터 코드 2/5: Document 구성 + FAISS + Kiwi BM25
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

# Documents
documents = []
for etf in etf_data:
    text = (f"{etf['name']} ({etf['category']}): {etf['description']} "
            f"키워드: {', '.join(etf['keywords'])} "
            f"수수료 {etf['expense_ratio']}% 배당 {etf['dividend_yield']}% "
            f"수익률 {etf['return_1y']:.1f}% 변동성 {etf['volatility']:.1f}%")
    documents.append(Document(page_content=text, metadata=etf))

# FAISS
vectorstore = LangchainFAISS.from_documents(documents, embeddings)
print(f"✅ FAISS: {vectorstore.index.ntotal}개 벡터")

# Kiwi 토크나이저
kiwi = Kiwi()

def kiwi_tokenize(text):
    """Kiwi 형태소 분석: NNG/NNP/SL/SN만 추출"""
    return [t.form.lower() for t in kiwi.tokenize(text) if t.tag in ("NNG", "NNP", "SL", "SN")]

# BM25
corpus = [kiwi_tokenize(doc.page_content) for doc in documents]
bm25 = BM25Okapi(corpus)

def bm25_search(query, k=5):
    """BM25 키워드 검색"""
    tokens = kiwi_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i].metadata["name"], scores[i]) for i in top_idx if scores[i] > 0]

print(f"✅ BM25: {len(corpus)}개 문서 인덱싱")

✅ FAISS: 12개 벡터
✅ BM25: 12개 문서 인덱싱


In [13]:
# ▶ 스타터 코드 3/5: 하이브리드 검색 + 메타데이터 필터
def hybrid_search(query, alpha=0.5, k=5):
    """벡터 + BM25 하이브리드 (Min-Max 정규화)"""
    vec_results = vectorstore.similarity_search_with_score(query, k=20)
    vec_scores = {doc.metadata["name"]: 1/(1+score) for doc, score in vec_results}

    tokens = kiwi_tokenize(query)
    bm25_raw = bm25.get_scores(tokens)
    bm25_scores = {documents[i].metadata["name"]: bm25_raw[i] for i in range(len(documents))}

    def minmax(d):
        vals = list(d.values())
        mn, mx = min(vals), max(vals)
        return {k: (v-mn)/(mx-mn+1e-8) for k, v in d.items()}

    vn, bn = minmax(vec_scores), minmax(bm25_scores)
    combined = {n: alpha*vn.get(n,0) + (1-alpha)*bn.get(n,0) for n in set(vn)|set(bn)}
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)[:k]

def filtered_search(vectorstore, query, filters=None, k=5, fetch_k=20):
    """벡터 검색 + 메타데이터 필터"""
    results = vectorstore.similarity_search_with_score(query, k=fetch_k)
    if not filters:
        return results[:k]
    filtered = []
    for doc, score in results:
        m = doc.metadata
        ok = True
        for key, val in filters.items():
            if isinstance(val, dict):
                if "less_than" in val and m.get(key, float("inf")) >= val["less_than"]:
                    ok = False
                if "greater_than" in val and m.get(key, 0) <= val["greater_than"]:
                    ok = False
            else:
                if m.get(key) != val:
                    ok = False
        if ok:
            filtered.append((doc, score))
    return filtered[:k]

# 평가용 쿼리셋
eval_queries = [
    {"query": "안전한 배당 ETF", "relevant": ["ACE 미국배당다우존스", "TIGER 국고채10년", "KODEX 단기채권PLUS"]},
    {"query": "미국 기술주 투자", "relevant": ["KODEX 미국나스닥100TR", "TIGER 미국반도체"]},
    {"query": "KODEX 200", "relevant": ["KODEX 200", "KODEX 200TR"]},
    {"query": "인플레이션 방어", "relevant": ["KODEX 골드선물(H)"]},
    {"query": "2차전지 테마", "relevant": ["TIGER 2차전지테마"]},
    {"query": "저비용 해외 ETF", "relevant": ["KODEX 미국S&P500TR", "KODEX 미국나스닥100TR"]},
    {"query": "고배당 저위험", "relevant": ["TIGER 고배당저변동", "ACE 미국배당다우존스"]},
    {"query": "부동산 투자", "relevant": ["TIGER 리츠부동산"]},
]

print(f"✅ 하이브리드 검색 + 필터 + 평가 쿼리 {len(eval_queries)}개 준비")

✅ 하이브리드 검색 + 필터 + 평가 쿼리 8개 준비


In [14]:
# ▶ 스타터 코드 4/5: 콘텐츠 기반 (CBF) + 협업 필터링 (CF)
risk_map = {"낮음": 1, "중간": 2, "높음": 3}

def cosine_sim(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-8))

def etf_to_vector(etf):
    return np.array([risk_map[etf["risk_level"]]/3, (etf["return_1y"]+30)/80,
                     etf["dividend_yield"]/6, etf["expense_ratio"]/1, etf["volatility"]/40])

item_vectors = np.array([etf_to_vector(e) for e in etf_data])
item_names = [e["name"] for e in etf_data]

# item-item 유사도 행렬
n = len(item_vectors)
item_sim = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        item_sim[i, j] = cosine_sim(item_vectors[i], item_vectors[j])

def cbf_similar_items(target_name, top_k=5):
    """item-item 코사인 유사도 Top-K"""
    idx = item_names.index(target_name)
    scores = [(item_names[j], item_sim[idx, j]) for j in range(n) if j != idx]
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# 협업 필터링
np.random.seed(42)
user_types = ["보수적_배당", "보수적_배당", "중립_인덱스", "중립_인덱스",
              "공격적_성장", "공격적_성장", "공격적_성장",
              "중립_균형", "배당매니아", "글로벌투자자"]

def seed_rating(ut, etf):
    cat = etf["category"]
    prefs = {
        "보수적_배당":  {"채권": 5, "배당": 5, "리츠": 4, "국내주식": 2, "해외주식": 1},
        "중립_인덱스":  {"국내주식": 5, "해외주식": 5, "배당": 3, "채권": 3, "리츠": 2, "원자재": 2},
        "공격적_성장":  {"해외주식": 5, "섹터": 5, "테마": 4, "원자재": 3, "국내주식": 3, "배당": 1},
        "배당매니아":   {"배당": 5, "리츠": 5, "채권": 3, "국내주식": 3, "해외주식": 2},
        "글로벌투자자": {"해외주식": 5, "원자재": 4, "섹터": 4, "국내주식": 2, "배당": 2, "테마": 3},
    }
    if ut == "중립_균형":
        return 3 + np.random.randint(-1, 2)
    return prefs.get(ut, {}).get(cat, 0)

R = np.zeros((len(user_types), len(etf_data)))
for u, ut in enumerate(user_types):
    for i, etf in enumerate(etf_data):
        r = seed_rating(ut, etf)
        if r > 0 and np.random.random() > 0.3:
            R[u, i] = r

# user-user 유사도
user_sim = np.zeros((R.shape[0], R.shape[0]))
for u in range(R.shape[0]):
    for v in range(R.shape[0]):
        if u == v: user_sim[u, v] = 1.0; continue
        mask = (R[u] > 0) & (R[v] > 0)
        user_sim[u, v] = cosine_sim(R[u][mask], R[v][mask]) if mask.sum() >= 2 else 0

def cf_recommend(user_idx, top_k=5):
    """User-based CF: 유사 사용자 가중평균"""
    sims = user_sim[user_idx]
    pred = np.zeros(R.shape[1])
    for i in range(R.shape[1]):
        if R[user_idx, i] > 0: pred[i] = -1; continue
        raters = np.where(R[:, i] > 0)[0]
        if len(raters) > 0 and sims[raters].sum() > 0:
            pred[i] = np.dot(sims[raters], R[raters, i]) / sims[raters].sum()
    top_idx = np.argsort(pred)[::-1][:top_k]
    return [(item_names[i], round(pred[i], 3)) for i in top_idx if pred[i] > 0]

print(f"✅ CBF: {n}×{n} 유사도 행렬 | CF: {R.shape} 평점 행렬 (밀도 {(R>0).sum()/R.size:.0%})")

✅ CBF: 12×12 유사도 행렬 | CF: (10, 12) 평점 행렬 (밀도 52%)


In [15]:
# ▶ 스타터 코드 5/5: LLM 추천 헬퍼
@dataclass
class InvestorProfile:
    name: str
    risk_tolerance: str
    investment_goal: str
    investment_horizon: int
    monthly_budget: int

def extract_profile(user_text):
    """자연어 → InvestorProfile"""
    prompt = f"""사용자 메시지에서 투자자 프로필을 JSON으로 추출하세요.
메시지: {user_text}
JSON: {{"name": "사용자", "risk_tolerance": "보수적|중립|공격적", "investment_goal": "안정수익|자산증식|배당수익", "investment_horizon": 숫자, "monthly_budget": 숫자}}"""
    resp = llm.invoke([{"role": "user", "content": prompt}]).content
    try:
        data = json.loads(resp)
    except Exception:
        match = re.search(r'\{{.*\}}', resp, re.DOTALL)
        data = json.loads(match.group()) if match else {}
    return InvestorProfile(
        name=data.get("name", "사용자"), risk_tolerance=data.get("risk_tolerance", "중립"),
        investment_goal=data.get("investment_goal", "자산증식"),
        investment_horizon=int(data.get("investment_horizon", 5)),
        monthly_budget=int(data.get("monthly_budget", 100)))

def rule_based_filter(risk_tolerance, etf_data):
    allowed = {"보수적": ["낮음"], "중립": ["낮음", "중간"], "공격적": ["낮음", "중간", "높음"]}
    return [e for e in etf_data if e["risk_level"] in allowed[risk_tolerance]]

def llm_recommend(profile, candidates, top_k=3):
    """LLM 추천 + allocation + reason"""
    etf_list = "\n".join([
        f"- {e['name']}: {e['category']}, 수익률 {e['return_1y']}%, 배당 {e['dividend_yield']}%, 위험 {e['risk_level']}, 수수료 {e['expense_ratio']}%"
        for e in candidates])
    prompt = f"""투자자: {profile.name} / {profile.risk_tolerance} / {profile.investment_goal} / {profile.investment_horizon}년 / 월 {profile.monthly_budget}만원
후보: {etf_list}
적합한 ETF {top_k}개를 JSON 배열로. [{{"name":"ETF명","allocation":비중(%),"reason":"이유"}}] 합계 100%. 순수 JSON만."""
    resp = llm.invoke([{"role": "system", "content": "ETF 투자 전문가."}, {"role": "user", "content": prompt}]).content
    try:
        return json.loads(resp)
    except Exception:
        match = re.search(r'\[.*\]', resp, re.DOTALL)
        return json.loads(match.group()) if match else []

print("✅ LLM 헬퍼 준비 완료")
print("\n🎯 스타터 코드 로드 완료! 아래 실습을 진행하세요.")

✅ LLM 헬퍼 준비 완료

🎯 스타터 코드 로드 완료! 아래 실습을 진행하세요.


---
### 실습 1: split() vs Kiwi 토큰화 품질 비교

수업에서 Kiwi 토크나이저를 구현했습니다. 이제 **왜 Kiwi가 split()보다 나은지** 정량적으로 증명하세요.

**구현할 것**:
1. `split_tokenize(text)` — 단순 공백 분리 + 소문자 변환
2. split 기반 BM25 인덱스 재구축
3. `eval_queries` 8개에 대해 **split BM25 vs Kiwi BM25** Hit Rate@3 비교
4. 실패하는 쿼리 예시와 이유 분석

> 핵심: "배당수익률" → split은 ["배당수익률"] (매칭 실패), Kiwi는 ["배당", "수익률"] (매칭 성공)

In [111]:
# 실습 1
def split_tokenize(text):
    """단순 공백 분리 토크나이저(+소문자 변환)"""
    splitted = text.lower().split(" ")
    return splitted

# split 기반 BM25 재구축
  # BM25Okapi(corpus_split): 토큰화된 문서 집합으로부터 BM25 점수를 계산할 수 있게 만드는 키워드 기반 lexical retriever / ranking model
corpus_split = [split_tokenize(doc.page_content) for doc in documents] # len(corpus_split) == 12
bm25_split = BM25Okapi(corpus_split) # 토큰화된 문서 집합(corpus_split)으로 BM25 검색 인덱스를 생성
                                    # 이후 query 토큰을 넣으면 각 문서와의 BM25 관련도 점수를 계산할 수 있음

def bm25_split_search(query, k=5):
    tokens = split_tokenize(query)
    scores = bm25_split.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    #return [(documents[i].metadata["name"], scores[i]) for i in top_idx if scores[i] > 0]
    return [(documents[i].metadata["name"], scores[i])]

# Hit Rate 비교
def hit_rate(eval_data, search_fn, k=5):
    hits = 0
    for item in eval_data:
        found = [r[0] for r in search_fn(item["query"], k=k)]
        if any(rel in found for rel in item["relevant"]):
            hits += 1
    return hits / len(eval_data)

# ---- 여기에 코드 작성 ----
  # split_hr = hit_rate(eval_queries, bm25_split_search, k=3)
  # kiwi_hr = hit_rate(eval_queries, bm25_search, k=3)
# 쿼리별 결과 비교표 출력
rows = []

for item in eval_queries:
    query = item.get('query')
    relevant = item.get('relevant')

    split_results = bm25_split_search(query, k=3)
    kiwi_results = bm25_search(query, k=3)

    split_hr3 = hit_rate([item], bm25_split_search, k=3)
    kiwi_hr3 = hit_rate([item], bm25_search, k=3)
    rows.append({
        "query": query,
        "relevant": ", ".join(relevant),
        "split_top3": ", ".join([r[0] for r in split_results]) if split_results else "[]",
        "kiwi_top3": ", ".join([r[0] for r in kiwi_results]) if kiwi_results else "[]",
        "hit_split@3": split_hr3,
        "hit_kiwi@3": kiwi_hr3,
    })

    print(f"query: {query}\nrelevant: {relevant}\n - hit_rate_split3: {split_hr3}\n - hit_rate_kiwi3: {kiwi_hr3}")
    print("="*20)

df_compare = pd.DataFrame(rows)
df_compare

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

query: 안전한 배당 ETF
relevant: ['ACE 미국배당다우존스', 'TIGER 국고채10년', 'KODEX 단기채권PLUS']
 - hit_rate_split3: 1.0
 - hit_rate_kiwi3: 1.0
query: 미국 기술주 투자
relevant: ['KODEX 미국나스닥100TR', 'TIGER 미국반도체']
 - hit_rate_split3: 0.0
 - hit_rate_kiwi3: 1.0
query: KODEX 200
relevant: ['KODEX 200', 'KODEX 200TR']
 - hit_rate_split3: 0.0
 - hit_rate_kiwi3: 1.0
query: 인플레이션 방어
relevant: ['KODEX 골드선물(H)']
 - hit_rate_split3: 0.0
 - hit_rate_kiwi3: 1.0
query: 2차전지 테마
relevant: ['TIGER 2차전지테마']
 - hit_rate_split3: 0.0
 - hit_rate_kiwi3: 1.0
query: 저비용 해외 ETF
relevant: ['KODEX 미국S&P500TR', 'KODEX 미국나스닥100TR']
 - hit_rate_split3: 0.0
 - hit_rate_kiwi3: 1.0
query: 고배당 저위험
relevant: ['TIGER 고배당저변동', 'ACE 미국배당다우존스']
 - hit_rate_split3: 1.0
 - hit_rate_kiwi3: 1.0
query: 부동산 투자
relevant: ['TIGER 리츠부동산']
 - hit_rate_split3: 0.0
 - hit_rate_kiwi3: 1.0


,query,relevant,split_top3,kiwi_top3,hit_split@3,hit_kiwi@3
0,안전한 배당 ETF,"ACE 미국배당다우존스, TIGER 국고채10년, KODEX 단기채권PLUS",ACE 미국배당다우존스,"TIGER 국고채10년, KODEX 단기채권PLUS, KODEX 골드선물(H)",1.0,1.0
1,미국 기술주 투자,"KODEX 미국나스닥100TR, TIGER 미국반도체",ACE 미국배당다우존스,"KODEX 미국나스닥100TR, TIGER 미국반도체, TIGER 2차전지테마",0.0,1.0
2,KODEX 200,"KODEX 200, KODEX 200TR",ACE 미국배당다우존스,"KODEX 200TR, KODEX 200",0.0,1.0
3,인플레이션 방어,KODEX 골드선물(H),ACE 미국배당다우존스,"KODEX 골드선물(H), TIGER 고배당저변동",0.0,1.0
4,2차전지 테마,TIGER 2차전지테마,ACE 미국배당다우존스,TIGER 2차전지테마,0.0,1.0
5,저비용 해외 ETF,"KODEX 미국S&P500TR, KODEX 미국나스닥100TR",ACE 미국배당다우존스,"KODEX 미국S&P500TR, KODEX 미국나스닥100TR",0.0,1.0
6,고배당 저위험,"TIGER 고배당저변동, ACE 미국배당다우존스",ACE 미국배당다우존스,"KODEX 단기채권PLUS, TIGER 2차전지테마, ACE 미국배당다우존스",1.0,1.0
7,부동산 투자,TIGER 리츠부동산,ACE 미국배당다우존스,"TIGER 리츠부동산, KODEX 200, TIGER 2차전지테마",0.0,1.0


#### 실패 쿼리 원인 분석
- 실패 쿼리 1: 핵심 키워드인 '해외'를 토큰화 하지 못함, kiwi의 경우 '해외주식'을 '해외', '주식'으로 토큰화 한 것으로 보임
  ```
  ====================
  query: 저비용 해외 ETF
  relevant: ['KODEX 미국S&P500TR', 'KODEX 미국나스닥100TR']
  - hit_rate_split3: 0.0
  - kiwi_hr3: 1.0
  ```
- 실패 쿼리 2: 문서에서 찾을 핵심 키워드인 '부동산' 이라는 단어를 제대로 토큰화 하지 못함 ('리츠부동산' -> '리츠', '부동산')
  ```
  ====================
  query: 부동산 투자
  relevant: ['TIGER 리츠부동산']
  - hit_rate_split3: 0.0
  - kiwi_hr3: 1.0
  ```

In [108]:
# 실패 쿼리 1 (split 기준)
  # bm25_split_search 함수의 score > 0 조건 해제 후 확인
# 실패이유: 핵심 키워드인 '해외'를 토큰화 하지 못함, kiwi의 경우 '해외주식'을 '해외', '주식'으로 토큰화 한 것으로 보임

q1 = "저비용 해외 ETF"

# 추가) query 토큰 / raw score / 상위 인덱스 확인
q1_split_tokens = split_tokenize(q1)
q1_scores = bm25_split.get_scores(q1_split_tokens)
q1_top_idx = np.argsort(q1_scores)[::-1][:10]

print(f"<토크나이저 비교>\n - 공백 split: {q1_split_tokens} \n - kiwi: {kiwi_tokenize(q1)}")
print(f"  - split top_idx: {q1_top_idx}")
print(f"  - split top_scores: {[q1_scores[i] for i in q1_top_idx]}")
print(f"""
bm25_split_search:{bm25_split_search(q1, k=3)}
bm25_search: {bm25_search(q1, k=3)}
""")

print("="*100)

# 추가 2) 상위 문서별 overlap / 토큰 / 원문 일부 확인
print("상위 문서별 overlap / 토큰 / 원문 일부 확인")
for i in q1_top_idx:
    doc_tokens = corpus_split[i]
    overlap = set(q1_split_tokens) & set(doc_tokens)

    print(f"[doc idx: {i}] name: {documents[i].metadata['name']}")
    print(f"score: {q1_scores[i]}")
    print(f"overlap: {overlap}")
    print(f"split query tokens: {q1_split_tokens}")
    print(f"doc_tokens[:30]: {doc_tokens[:30]}")
    print("-" * 60)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<토크나이저 비교>
 - 공백 split: ['저비용', '해외', 'etf'] 
 - kiwi: ['비용', '해외', 'etf']
  - split top_idx: [11 10  9  8  7  6  5  4  3  2]
  - split top_scores: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
 
bm25_split_search:[('TIGER 미국반도체', np.float64(0.0))]
bm25_search: [('KODEX 미국S&P500TR', np.float64(3.765543210237243)), ('KODEX 미국나스닥100TR', np.float64(2.0079911341156653))]

상위 문서별 overlap / 토큰 / 원문 일부 확인
[doc idx: 11] name: TIGER 고배당저변동
score: 0.0
overlap: set()
split query tokens: ['저비용', '해외', 'etf']
doc_tokens[:30]: ['tiger', '고배당저변동', '(배당):', '고배당+저변동성', '종목을', '선별.', '배당수익률', '5.2%로', '방어적', '포트폴리오에', '적합합니다.', '키워드:', '고배당,', '저변동,', '방어적,', '배당,', '안정', '수수료', '0.3%', '배당', '5.2%', '수익률', '7.2%', '변동성', '8.7%']
------------------------------------------------------------
[doc idx: 10] name: KODEX 200TR
score: 0.0
overlap: set()
split query tokens: ['저비용', '해외',

In [106]:
# q1 = "저비용 해외 ETF" 에 대해 kiwi 기준 디버깅

# 1) kiwi 기준 코퍼스와 BM25 객체 준비
corpus_kiwi = [kiwi_tokenize(doc.page_content) for doc in documents]
bm25_kiwi = BM25Okapi(corpus_kiwi)

# 2) query를 kiwi로 토큰화하고 score 계산
q1 = "저비용 해외 ETF"
q1_kiwi_tokens = kiwi_tokenize(q1)
q1_kiwi_scores = bm25_kiwi.get_scores(q1_kiwi_tokens)
q1_kiwi_top_idx = np.argsort(q1_kiwi_scores)[::-1][:5]

print(f"<kiwi 디버그>")
print(f"kiwi query tokens: {q1_kiwi_tokens}")
print(f"kiwi top_idx: {q1_kiwi_top_idx}")
print(f"kiwi top_scores: {[q1_kiwi_scores[i] for i in q1_kiwi_top_idx]}")
print(f"bm25_search(q1, k=3): {bm25_search(q1, k=3)}")

print("=" * 100)
print("kiwi 상위 문서별 overlap / 토큰 / 원문 일부 확인")

for i in q1_kiwi_top_idx:
    doc_tokens = corpus_kiwi[i]
    overlap = set(q1_kiwi_tokens) & set(doc_tokens)

    print(f"[doc idx: {i}] name: {documents[i].metadata['name']}")
    print(f"score: {q1_kiwi_scores[i]}")
    print(f"overlap: {overlap}")
    print(f"kiwi query tokens: {q1_kiwi_tokens}")
    print(f"kiwi doc_tokens[:30]: {doc_tokens[:30]}")
    print(f"page_content[:200]: {documents[i].page_content[:200]}")
    print("-" * 60)

print("all_nonzero_scores:")
print([(i, documents[i].metadata["name"], q1_kiwi_scores[i]) for i in range(len(q1_kiwi_scores)) if q1_kiwi_scores[i] > 0])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<kiwi 디버그>
kiwi query tokens: ['비용', '해외', 'etf']
kiwi top_idx: [ 1  6 11 10  8]
kiwi top_scores: [np.float64(3.765543210237243), np.float64(2.0079911341156653), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
bm25_search(q1, k=3): [('KODEX 미국S&P500TR', np.float64(3.765543210237243)), ('KODEX 미국나스닥100TR', np.float64(2.0079911341156653))]
kiwi 상위 문서별 overlap / 토큰 / 원문 일부 확인
[doc idx: 1] name: KODEX 미국S&P500TR
score: 3.765543210237243
overlap: {'비용', '해외'}
kiwi query tokens: ['비용', '해외', 'etf']
kiwi doc_tokens[:30]: ['kodex', '미국', 's', 'p', '500', 'tr', '해외', '주식', '미국', 's', 'p', '500', '지수', '수익', 'tr', '추종', '비용', '미국', '대형주', '전체', '분산', '투자', '키워드', 's', 'p', '500', '미국', '대형주', '패시브', '해외']
page_content[:200]: KODEX 미국S&P500TR (해외주식): 미국 S&P500 지수의 총수익(TR)을 추종. 저비용으로 미국 대형주 전체에 분산투자합니다. 키워드: S&P500, 미국, 대형주, 패시브, 해외주식 수수료 0.05% 배당 0.0% 수익률 25.3% 변동성 18.5%
------------------------------------------------------------
[doc idx: 6] name: KODEX 미국나스닥100TR
score: 2.0079911341156653
o

In [105]:
# 실패 쿼리 2
  # bm25_split_search 함수의 score > 0 조건 해제 후 확인
# 실패 이유: 문서에서 찾을 핵심 키워드인 '부동산' 이라는 단어를 제대로 토큰화 하지 못함
  # 예: '리츠부동산','리츠·부동산'

q2 = "부동산 투자"

q2_split_tokens = split_tokenize(q2)
q2_scores = bm25_split.get_scores(q2_split_tokens)
q2_top_idx = np.argsort(q2_scores)[::-1][:5]

print(f"<토크나이저 비교>\n - 공백 split: {q2_split_tokens} \n - kiwi: {kiwi_tokenize(q2)}")
print(f" - split top_idx: {q2_top_idx}")
print(f" - split top_scores: {[q2_scores[i] for i in q2_top_idx]}")
print(f"""
bm25_split_search: {bm25_split_search(q2, k=3)}
bm25_search: {bm25_search(q2, k=3)}
""")

print("=" * 100)

print("상위 문서별 overlap / 토큰 / 원문 일부 확인")
for i in q2_top_idx:
    doc_tokens = corpus_split[i]
    overlap = set(q2_split_tokens) & set(doc_tokens)

    print(f"[doc idx: {i}] name: {documents[i].metadata['name']}")
    print(f"score: {q2_scores[i]}")
    print(f"overlap: {overlap}")
    print(f"split query tokens: {q2_split_tokens}")
    print(f"doc_tokens[:30]: {doc_tokens[:30]}")
    print("-" * 60)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<토크나이저 비교>
 - 공백 split: ['부동산', '투자'] 
 - kiwi: ['부동산', '투자']
 - split top_idx: [11 10  9  8  7]
 - split top_scores: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
 
bm25_split_search: [('TIGER 미국반도체', np.float64(0.0))]
bm25_search: [('TIGER 리츠부동산', np.float64(3.733570658386896)), ('KODEX 200', np.float64(0.4427069320427485)), ('TIGER 2차전지테마', np.float64(0.30548884541692295))]

상위 문서별 overlap / 토큰 / 원문 일부 확인
[doc idx: 11] name: TIGER 고배당저변동
score: 0.0
overlap: set()
split query tokens: ['부동산', '투자']
doc_tokens[:30]: ['tiger', '고배당저변동', '(배당):', '고배당+저변동성', '종목을', '선별.', '배당수익률', '5.2%로', '방어적', '포트폴리오에', '적합합니다.', '키워드:', '고배당,', '저변동,', '방어적,', '배당,', '안정', '수수료', '0.3%', '배당', '5.2%', '수익률', '7.2%', '변동성', '8.7%']
------------------------------------------------------------
[doc idx: 10] name: KODEX 200TR
score: 0.0
overlap: set()
split query tokens: ['부동산', '투자']
doc_tokens[:30]: ['kodex', '200tr', '(국내주식):', 'kodex', '200의', '총수익(tr)', '버전.',

#### 출력 확인용 섹션

In [69]:
eval_queries[0].get('query')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'안전한 배당 ETF'

In [55]:
def split_tokenize(text):
    """단순 공백 분리 토크나이저"""
    splitted = text.lower().split(" ")
    return splitted
corpus_split = [split_tokenize(doc.page_content) for doc in documents]
len(corpus_split)
corpus_split

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[['kodex',
  '200',
  '(국내주식):',
  'kospi',
  '200',
  '지수를',
  '추종하는',
  '대표',
  '국내',
  'etf.',
  '대형주',
  '중심',
  '분산투자에',
  '적합합니다.',
  '키워드:',
  '코스피,',
  '대형주,',
  '인덱스,',
  '분산투자,',
  '국내주식',
  '수수료',
  '0.15%',
  '배당',
  '1.8%',
  '수익률',
  '8.5%',
  '변동성',
  '15.2%'],
 ['kodex',
  '미국s&p500tr',
  '(해외주식):',
  '미국',
  's&p500',
  '지수의',
  '총수익(tr)을',
  '추종.',
  '저비용으로',
  '미국',
  '대형주',
  '전체에',
  '분산투자합니다.',
  '키워드:',
  's&p500,',
  '미국,',
  '대형주,',
  '패시브,',
  '해외주식',
  '수수료',
  '0.05%',
  '배당',
  '0.0%',
  '수익률',
  '25.3%',
  '변동성',
  '18.5%'],
 ['ace',
  '미국배당다우존스',
  '(배당):',
  '미국',
  '고배당',
  '대형주',
  '중심.',
  '최저',
  '수수료(0.01%)로',
  '안정적',
  '배당수익을',
  '추구합니다.',
  '키워드:',
  '미국배당,',
  '다우존스,',
  '고배당,',
  '월배당,',
  '안정',
  '수수료',
  '0.01%',
  '배당',
  '3.5%',
  '수익률',
  '12.1%',
  '변동성',
  '10.3%'],
 ['tiger',
  '2차전지테마',
  '(테마):',
  '2차전지·배터리',
  '관련',
  '기업에',
  '집중',
  '투자.',
  '고성장',
  '기대',
  '대신',
  '높은',
  '변동성을',
  '감수해야',
  '합니다.',
  '키워드:',
  '2차전지,',
  '배터리,'

In [56]:
# Kiwi 토크나이저 (위에 이미 존재하는 코드)
kiwi = Kiwi()

def kiwi_tokenize(text):
    """Kiwi 형태소 분석: NNG/NNP/SL/SN만 추출"""
    return [t.form.lower() for t in kiwi.tokenize(text) if t.tag in ("NNG", "NNP", "SL", "SN")]

# BM25
corpus = [kiwi_tokenize(doc.page_content) for doc in documents]
bm25 = BM25Okapi(corpus)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
corpus_split

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[['KODEX',
  '200',
  '(국내주식):',
  'KOSPI',
  '200',
  '지수를',
  '추종하는',
  '대표',
  '국내',
  'ETF.',
  '대형주',
  '중심',
  '분산투자에',
  '적합합니다.',
  '키워드:',
  '코스피,',
  '대형주,',
  '인덱스,',
  '분산투자,',
  '국내주식',
  '수수료',
  '0.15%',
  '배당',
  '1.8%',
  '수익률',
  '8.5%',
  '변동성',
  '15.2%'],
 ['KODEX',
  '미국S&P500TR',
  '(해외주식):',
  '미국',
  'S&P500',
  '지수의',
  '총수익(TR)을',
  '추종.',
  '저비용으로',
  '미국',
  '대형주',
  '전체에',
  '분산투자합니다.',
  '키워드:',
  'S&P500,',
  '미국,',
  '대형주,',
  '패시브,',
  '해외주식',
  '수수료',
  '0.05%',
  '배당',
  '0.0%',
  '수익률',
  '25.3%',
  '변동성',
  '18.5%'],
 ['ACE',
  '미국배당다우존스',
  '(배당):',
  '미국',
  '고배당',
  '대형주',
  '중심.',
  '최저',
  '수수료(0.01%)로',
  '안정적',
  '배당수익을',
  '추구합니다.',
  '키워드:',
  '미국배당,',
  '다우존스,',
  '고배당,',
  '월배당,',
  '안정',
  '수수료',
  '0.01%',
  '배당',
  '3.5%',
  '수익률',
  '12.1%',
  '변동성',
  '10.3%'],
 ['TIGER',
  '2차전지테마',
  '(테마):',
  '2차전지·배터리',
  '관련',
  '기업에',
  '집중',
  '투자.',
  '고성장',
  '기대',
  '대신',
  '높은',
  '변동성을',
  '감수해야',
  '합니다.',
  '키워드:',
  '2차전지,',
  '배터리,'

In [48]:
corpus_split = [split_tokenize(doc.page_content) for doc in documents]
bm25_split = BM25Okapi(corpus_split)
bm25_split

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
bm25_split.corpus_size   # 문서 수
bm25_split.avgdl         # 평균 문서 길이
bm25_split.doc_len[:5]   # 앞 5개 문서 길이
bm25_split.idf           # 토큰별 idf 사전

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'KODEX': 0.0,
 '200': 2.0368819272610397,
 '(국내주식):': 1.4350845252893225,
 'KOSPI': 2.0368819272610397,
 '지수를': 1.4350845252893225,
 '추종하는': 1.4350845252893225,
 '대표': 2.0368819272610397,
 '국내': 1.4350845252893225,
 'ETF.': 0.6359887667199966,
 '대형주': 0.9985288301111273,
 '중심': 1.4350845252893225,
 '분산투자에': 2.0368819272610397,
 '적합합니다.': 1.4350845252893225,
 '키워드:': 0.44894649708078505,
 '코스피,': 2.0368819272610397,
 '대형주,': 1.4350845252893225,
 '인덱스,': 1.4350845252893225,
 '분산투자,': 2.0368819272610397,
 '국내주식': 1.4350845252893225,
 '수수료': 0.44894649708078505,
 '0.15%': 2.0368819272610397,
 '배당': 0.44894649708078505,
 '1.8%': 2.0368819272610397,
 '수익률': 0.44894649708078505,
 '8.5%': 2.0368819272610397,
 '변동성': 0.44894649708078505,
 '15.2%': 2.0368819272610397,
 '미국S&P500TR': 2.0368819272610397,
 '(해외주식):': 1.4350845252893225,
 '미국': 0.9985288301111273,
 'S&P500': 2.0368819272610397,
 '지수의': 2.0368819272610397,
 '총수익(TR)을': 2.0368819272610397,
 '추종.': 1.4350845252893225,
 '저비용으로': 2.0368

---
### 실습 2: 쿼리 의도 분류 + 스마트 검색 라우터

자연어 쿼리를 **조건 / 키워드 / 의미** 3가지로 분류하고, 적합한 검색 전략을 자동으로 선택하는 `smart_router`를 구축하세요.

**구현할 것**:
1. `detect_intent(query)` — rule 기반 분류기: 숫자/% → "조건", 브랜드명 → "키워드", 나머지 → "의미"
2. `extract_filters(query)` — LLM으로 자연어에서 필터 JSON 추출
3. `smart_router(query)` — 의도에 따라 filtered_search / bm25_search / hybrid_search 자동 선택

In [126]:
etf_data[0]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'name': 'KODEX 200',
 'category': '국내주식',
 'expense_ratio': 0.15,
 'return_1y': 8.5,
 'risk_level': '중간',
 'dividend_yield': 1.8,
 'volatility': 15.2,
 'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식'],
 'description': 'KOSPI 200 지수를 추종하는 대표 국내 ETF. 대형주 중심 분산투자에 적합합니다.'}

In [127]:
def _build_keyword_terms(etf_data):
    keyword_terms = set()

    generic_weak_terms = {
        "미국", "국내", "해외", "배당", "채권", "성장", "안정", "테마",
        "섹터", "원자재", "리츠", "대형주", "인덱스", "분산투자", "기술",
        "안전자산", "고위험", "저위험"
    }

    for item in etf_data:
        name = item["name"].lower()
        keywords = [k.lower() for k in item.get("keywords", [])]

        # 1) 브랜드명: name의 첫 토큰
        first_token = name.split()[0]
        keyword_terms.add(first_token)

        # 2) 주식 이름 전체
        keyword_terms.add(name)

        # 3) name 안의 세부 토큰
        for tok in re.split(r"[\/\s\-\+\(\)·]+", name):
            tok = tok.strip()
            if tok and tok not in generic_weak_terms and len(tok) >= 2:
                keyword_terms.add(tok)

    return keyword_terms


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [135]:
print(detect_intent("수수료 0.1% 이하이면서 배당 3% 이상인 ETF"))
print(detect_intent("TIGER 나스닥과 관련된 주식을 추천해줘"))
print(detect_intent("초보자에게 좋은 상품은?"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

조건
키워드
의미


In [142]:
# 실습 2
def detect_intent(query):
    """[Rule 기반 분류] 쿼리 → '조건(숫자/%)' | '키워드(브랜드명)' | '의미(나머지)'"""
    # 쿼리의 의도를 어떻게 구별할 수 있을지? => Rule 기반 분류임
      # '조건'의 의도 쿼리 예문: "수수료 0.1% 이하이면서 배당 3% 이상인 ETF"

    CONDITION_PATTERNS = [
      (r'\d+(\.\d+)?\s*%', 3),                       # "0.1%", "3%"
      (r'(이하|이상|미만|초과|넘는|넘지\s*않는)', 2), # 비교 연산자
      (r'\d+(\.\d+)?\s*(원|만원|달러)', 2),           # 금액
    ]
    keyword_terms = _build_keyword_terms(etf_data)

    def _score_condition(query: str) -> int:
      return sum(w for pat, w in CONDITION_PATTERNS if re.search(pat, query))

    def _score_keyword(query: str, keyword_terms: set) -> int:
      # keyword_terms가 lower-case로 구성되어 있으므로 쿼리도 소문자화
      q = query.lower()
      return sum(1 for term in keyword_terms if term in q)

    cond = _score_condition(query)
    kw   = _score_keyword(query, _build_keyword_terms(etf_data))

    # 임계값 기반 우선순위 (조건 > 키워드 > 의미)
    if cond >= 2: return '조건'
    if kw   >= 1: return '키워드'

    return '의미' # 어느 쪽도 아니면 '의미'로 분류


def extract_filters(query):
    """LLM으로 필터 JSON 추출"""
    prompt = f"""사용자 query에서 검색 필터를 추출하여 JSON 형식으로 작성하세요.

    쿼리 : {query}

    사용 가능한 필터 필드:
    - category :
    - expense_ratio : 숫자 (%)
    - return_1y : 숫자 (%)
    - risk_level : 낮음, 중간, 높음
    - dividend_yield : 숫자 (%)
    - volatility : 숫자 (%)

    JSON 형식으로만 응답하세요. 필터가 없으면 {{}}
    예, {{"category": "해외주식", "expense_ratio":}}
    """

    response = llm.invoke([
        {'role': 'user',
         'content': prompt
         }
    ])
    try:
      if "```" in response:
        response = response.split("```")[1].replace("json","") # ``` 으로 감싸진 부분 제거
      return json.loads(response)
    except Exception:
      return {}

def smart_router(query, k=3):
    """의도 분류 → 적절한 검색 함수 선택 → 결과 반환"""
    #의도에 따라 검색 함수 *자동 선택*
    intent = detect_intent(query)
    # ---- 여기에 코드 작성 ----

    if intent == '조건':
    # 조건 → extract_filters + filtered_search
        filters = extract_filters(query)
        results = filtered_search(vectorstore, query, filters=filters, k=5, fetch_k=20)
        return [(doc.metadata['name'], score) for doc, score in results]

    elif intent == '키워드':
    # 키워드 → bm25_search
        results = bm25_search(query)
        return results

    else:
    # 의미 → hybrid_search
        results = hybrid_search(query)
        return results


# 테스트
test_queries = ["수수료 0.1% 이하 ETF", "KODEX 200", "안전한 장기투자",
                "배당 3% 이상 해외 ETF", "2차전지 관련 성장주"]
for q in test_queries:
    intent = detect_intent(q)
    results = smart_router(q, k=3)
    names = [r[0] if isinstance(r, tuple) else r.metadata["name"] for r in (results or [])]
    print(f"[{intent:3s}] {q:25s} → {names}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[조건 ] 수수료 0.1% 이하 ETF           → ['TIGER 국고채10년', 'KODEX 미국나스닥100TR', 'KODEX 200', 'KODEX 단기채권PLUS', 'KODEX 미국S&P500TR']
[키워드] KODEX 200                 → ['KODEX 200TR', 'KODEX 200']
[의미 ] 안전한 장기투자                  → ['TIGER 국고채10년', 'KODEX 단기채권PLUS', 'KODEX 200TR', 'TIGER 2차전지테마', 'TIGER 리츠부동산']
[조건 ] 배당 3% 이상 해외 ETF           → ['KODEX 미국나스닥100TR', 'TIGER 국고채10년', 'KODEX 미국S&P500TR', 'KODEX 200', 'ACE 미국배당다우존스']
[의미 ] 2차전지 관련 성장주               → ['TIGER 2차전지테마', 'KODEX 미국나스닥100TR', 'TIGER 고배당저변동', 'KODEX 단기채권PLUS', 'TIGER 리츠부동산']


---
### 실습 3: 하이브리드 α 최적화 + 3방식 비교표

`hybrid_search`의 α를 0.0~1.0으로 스윕하여 **최적 alpha**를 찾고, FAISS / BM25 / Hybrid(최적α) 3방식의 성능을 비교하세요.

**구현할 것**:
1. Hit Rate@3 기준 최적 α 탐색 (0.1 단위)
2. 3방식 비교표 출력 (HR@3, HR@5)
3. α=0(BM25 전용)과 α=1(벡터 전용)에서 각각 실패하는 쿼리 분석

In [151]:

# 평가용 쿼리셋
eval_queries = [
    {"query": "안전한 배당 ETF", "relevant": ["ACE 미국배당다우존스", "TIGER 국고채10년", "KODEX 단기채권PLUS"]},
    {"query": "미국 기술주 투자", "relevant": ["KODEX 미국나스닥100TR", "TIGER 미국반도체"]},
    {"query": "KODEX 200", "relevant": ["KODEX 200", "KODEX 200TR"]},
    {"query": "인플레이션 방어", "relevant": ["KODEX 골드선물(H)"]},
    {"query": "2차전지 테마", "relevant": ["TIGER 2차전지테마"]},
    {"query": "저비용 해외 ETF", "relevant": ["KODEX 미국S&P500TR", "KODEX 미국나스닥100TR"]},
    {"query": "고배당 저위험", "relevant": ["TIGER 고배당저변동", "ACE 미국배당다우존스"]},
    {"query": "부동산 투자", "relevant": ["TIGER 리츠부동산"]},
]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [158]:
# 실습 3

# 1) alpha 스윕 → 최적 alpha 탐색
best_alpha, best_hr = 0, 0
alpha_results = []
for alpha in np.arange(0, 1.1, 0.1):
    # hybrid_fn을 alpha로 만들고 hit_rate 계산
    hybrid_fn = lambda q, k=2, a=alpha: hybrid_search(q, alpha=a, k=k)
    hr2 = hit_rate(eval_queries, hybrid_fn, k=2)
    alpha_results.append((round(alpha, 1), hr2))
    if hr2> best_hr:
        best_hr, best_alpha = hr2, round(alpha, 1)

print("α 스윕 (HR@2):")
for a, hr in alpha_results:
    mark = " ← best" if a == best_alpha else ""
    print(f"  α={a:.1f} | HR@2 = {hr:.3f}{mark}")
print(f"\n✅ 최적 α = {best_alpha} (HR@2 = {best_hr:.3f})\n")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

α 스윕 (HR@2):
  α=0.0 | HR@2 = 0.875
  α=0.1 | HR@2 = 0.875
  α=0.2 | HR@2 = 0.875
  α=0.3 | HR@2 = 0.875
  α=0.4 | HR@2 = 0.875
  α=0.5 | HR@2 = 1.000 ← best
  α=0.6 | HR@2 = 1.000
  α=0.7 | HR@2 = 1.000
  α=0.8 | HR@2 = 1.000
  α=0.9 | HR@2 = 1.000
  α=1.0 | HR@2 = 1.000

✅ 최적 α = 0.5 (HR@2 = 1.000)



In [159]:

# 2) 3방식 비교표
def vec_fn(q, k=5):
    r = vectorstore.similarity_search(q, k=k)
    return [(d.metadata["name"], 1.0) for d in r]


def hybrid_best_fn(q, k=5, a=best_alpha):
    return hybrid_search(q, alpha=a, k=k)

methods = [
    ("FAISS(벡터)",             vec_fn),
    ("BM25",                    bm25_search),
    (f"Hybrid(α={best_alpha})", hybrid_best_fn),
]
print(f"{'방법':20s} | HR@2   | HR@5")
print("-" * 42)
for name, fn in methods:
    hr2 = hit_rate(eval_queries, fn, k=2)
    hr5 = hit_rate(eval_queries, fn, k=5)

    print(f"{name:20s} | {hr2:.3f}  | {hr5:.3f}")

# 3) α=0 vs α=1 실패 쿼리 분석

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

방법                   | HR@2   | HR@5
------------------------------------------
FAISS(벡터)            | 1.000  | 1.000
BM25                 | 0.875  | 1.000
Hybrid(α=0.5)        | 1.000  | 1.000


In [160]:
# 3) α=0 vs α=1 실패 쿼리 분석
# 원인: 동의어 문제 ( Q: 고배당 저위험, A: 고배당저변동)
bm25_fail_k2 = []
for ex in eval_queries:
    retrieved = [name for name, _ in hybrid_search(ex["query"], alpha=0.0, k=2)]
    if not any(rel in retrieved for rel in ex["relevant"]):
        bm25_fail_k2.append((ex["query"], ex["relevant"], retrieved))

print(f"α=0 (BM25) HR@2 실패: {len(bm25_fail_k2)}개")
for q, rel, ret in bm25_fail_k2:
    print(f"  Q: {q}")
    print(f"    정답: {rel}")
    print(f"    Top2: {ret}\n")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

α=0 (BM25) HR@2 실패: 1개
  Q: 고배당 저위험
    정답: ['TIGER 고배당저변동', 'ACE 미국배당다우존스']
    Top2: ['KODEX 단기채권PLUS', 'TIGER 2차전지테마']



---
### 실습 4: MAP@K + LLM-as-Judge 검색 품질 평가

수업에서 Hit Rate/MRR을 배웠으니, 이번엔 **새로운 메트릭**으로 검색 품질을 평가하세요.

**구현할 것**:
1. `average_precision(ranked_list, relevant_set)` — 순위별 precision 누적 평균
2. `map_at_k(eval_queries, search_fn, k)` — 전체 쿼리의 AP 평균 (Mean Average Precision)
3. `llm_judge_search(query, results, k)` — LLM이 각 검색 결과의 관련성을 1~5점 채점
4. 3방식 × (MAP@5 + LLM Judge 평균) 비교표

> **MAP@K 공식**: AP(q) = (1/min(K, |Rel|)) × Σ [Precision@i × rel(i)]
> **LLM Judge**: "이 쿼리에 대해 이 ETF가 얼마나 관련있나요? 1~5점"

In [163]:
# 실습 4
def average_precision(ranked_names, relevant_set, k=5):
    """단일 쿼리의 Average Precision@K"""
    # ---- 여기에 코드 작성 ----
    ranked = ranked_names[:k]
    if not relevant_set:
        return 0.0

    hits, score = 0, 0.0
    for i, name in enumerate(ranked, start=1):  # 1-based
        if name in relevant_set:
            hits += 1
            score += hits / i    # Precision@i (정답일 때만 누적)
    denom = min(k, len(relevant_set))

    return score / denom if denom > 0 else 0.0

def map_at_k(eval_data, search_fn, k=5):
    """Mean Average Precision@K"""
    aps = []
    for item in eval_data:
        ranked = [r[0] for r in search_fn(item["query"], k=k)]
        aps.append(average_precision(ranked, set(item["relevant"]), k=k))

    return np.mean(aps)

def llm_judge_search(query, results, k=5):
    """LLM이 검색 결과의 관련성을 1~5점 채점"""
    # ---- 여기에 코드 작성 ----
    # 각 결과에 대해 LLM에게 "쿼리: {query}, ETF: {name} — 관련성 1~5점?" 질문
    # JSON으로 [{name: score}, ...] 반환
    names = [name for name, _ in results[:k]]
    prompt = f"""당신은 ETF 추천 품질을 평가하는 전문 심사자입니다.
      사용자 쿼리와 검색 결과 ETF 이름이 주어졌을 때, 각 ETF가 쿼리에 얼마나 관련 있는지 1~5점으로 채점하세요.

      채점 기준:
        5 = 완벽히 관련 (쿼리 의도 정확 일치)
        4 = 매우 관련 (핵심 속성 대부분 일치)
        3 = 보통 (일부 속성 일치)
        2 = 약한 관련 (카테고리만 겹침)
        1 = 무관

      쿼리: "{query}"
      검색 결과 ETF:
      {chr(10).join(f"- {n}" for n in names)}

      다음 JSON 형식으로만 답하세요 (다른 설명 금지):
      {{"scores": {{"ETF이름1": 점수, "ETF이름2": 점수, ...}}}}
    """
    response = llm.invoke(prompt).content

    # JSON 추출
    m = re.search(r"\{.*\}", response, flags=re.DOTALL)
    data = json.loads(m.group(0)) if m else {"scores": {}}

    return data.get("scores", {})


    pass

# 3방식 × MAP@5 비교
for name, fn in [("FAISS", vec_fn), ("BM25", bm25_search), ("Hybrid", hybrid_best_fn)]:
    print(f"{name}: MAP@5={map_at_k(eval_queries, fn, 5):.3f}")

# LLM Judge 샘플 (2~3개 쿼리만)
for q in ["안전한 배당 ETF", "미국 기술주 투자"]:
    scores = llm_judge_search(q, hybrid_search(q, alpha=0.5, k=3))
    print(f"  {q} → {scores}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FAISS: MAP@5=0.958
BM25: MAP@5=0.917
Hybrid: MAP@5=0.921
  안전한 배당 ETF → {'TIGER 국고채10년': 4, 'KODEX 단기채권PLUS': 4, 'KODEX 골드선물(H)': 1}
  미국 기술주 투자 → {'KODEX 미국나스닥100TR': 5, 'TIGER 미국반도체': 4, 'TIGER 2차전지테마': 2}


---
### 실습 5: CBF 다양성 확보 + Cold-Start 폴백

`cbf_similar_items`는 같은 카테고리만 추천할 수 있습니다. **카테고리 중복 없이** 다양한 추천을 하세요.
또한 **신규 사용자(평점 0개)** 일 때 CF 대신 CBF로 폴백하는 로직을 구현하세요.

**구현할 것**:
1. `cbf_diverse(target_name, top_k)` — 카테고리 중복 없이 Top-K 선택
2. `cold_start_recommend(R, user_idx, favorite_etf, top_k)` — 평점 < 2개면 CBF 폴백, 아니면 CF

In [146]:
target_name = 'KODEX 200'
idx = item_names.index(target_name)
scored = [(j, item_sim[idx, j]) for j in range(n) if j != idx]
scored

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[(1, np.float64(0.9317615934699991)),
 (2, np.float64(0.8708104267143443)),
 (3, np.float64(0.843935304857269)),
 (4, np.float64(0.8891845432574942)),
 (5, np.float64(0.8626763097569392)),
 (6, np.float64(0.9417740369716172)),
 (7, np.float64(0.9242601096073657)),
 (8, np.float64(0.8240899758256612)),
 (9, np.float64(0.9101145298708155)),
 (10, np.float64(0.9499766521932036)),
 (11, np.float64(0.7827706706731448))]

In [149]:
# 실습 5

# item_sim = np.zeros((n, n))
# for i in range(n):
#     for j in range(n):
#         item_sim[i, j] = cosine_sim(item_vectors[i], item_vectors[j])

# item_names = [e["name"] for e in etf_data]

def cbf_diverse(target_name, top_k=5): # target_name : 예) KODEX 200 등
    """카테고리 중복 없이 유사 ETF 추천"""
    idx = item_names.index(target_name)
    scored = [(j, item_sim[idx, j]) for j in range(n) if j != idx]
    scored.sort(key=lambda x: x[1], reverse=True)
    # ---- 여기에 코드 작성 ----
    # seen_cats 집합으로 카테고리 중복 제거
    result = []
    seen_cats = set() # 이미 나온 카테고리

    for j, score in scored:
      cat = etf_data[j]['category']
      if cat in seen_cats:
        continue
      result.append((item_names[j], round(score,3), cat)) # 이름 및 카테고리를 result = []에 추가
      seen_cats.add(cat)
      if len(result) >= top_k:
        break

    return result

def cold_start_recommend(R, user_idx, favorite_etf=None, top_k=5):
    """신규 사용자면 CBF 폴백, 기존 사용자면 CF"""
    # 신규 사용자(평점 0개) 일 때 CF 대신 CBF로 폴백하는 로직

    is_new_user = (user_idx >= R.shape[0]) or (R[user_idx].sum() == 0)

    if is_new_user:
      if favorite_etf is None:
        return []
      return cbf_diverse(favorite_etf, top_k=top_k)

    else:
       return cf_recommend(user_idx, top_k=top_k)

# 테스트
print("CBF 일반:")
for name, score in cbf_similar_items("KODEX 200", top_k=5):
    etf = next(e for e in etf_data if e["name"] == name)
    print(f"  {score:.3f} | {name} ({etf['category']})")

print("\nCBF 다양성:")
for row in cbf_diverse("KODEX 200", top_k=5):
    print(f"  {row}")

print("\nCold-start (신규, fav=KODEX 200):")
print(cold_start_recommend(R, user_idx=99, favorite_etf="KODEX 200", top_k=3))
print("기존 사용자 U0:")
print(cold_start_recommend(R, user_idx=0, top_k=3))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CBF 일반:
  0.950 | KODEX 200TR (국내주식)
  0.942 | KODEX 미국나스닥100TR (해외주식)
  0.932 | KODEX 미국S&P500TR (해외주식)
  0.924 | TIGER 미국반도체 (섹터)
  0.910 | TIGER 리츠부동산 (리츠)

CBF 다양성:
  ('KODEX 200TR', np.float64(0.95), '국내주식')
  ('KODEX 미국나스닥100TR', np.float64(0.942), '해외주식')
  ('TIGER 미국반도체', np.float64(0.924), '섹터')
  ('TIGER 리츠부동산', np.float64(0.91), '리츠')
  ('TIGER 국고채10년', np.float64(0.889), '채권')

Cold-start (신규, fav=KODEX 200):
[('KODEX 200TR', np.float64(0.95), '국내주식'), ('KODEX 미국나스닥100TR', np.float64(0.942), '해외주식'), ('TIGER 미국반도체', np.float64(0.924), '섹터')]
기존 사용자 U0:
[('KODEX 미국나스닥100TR', np.float64(4.181)), ('TIGER 2차전지테마', np.float64(3.629)), ('TIGER 리츠부동산', np.float64(3.354))]


#### 기존 코드

In [ ]:
def cf_recommend(user_idx, top_k=5):
    """User-based CF: 유사 사용자 가중평균"""
    sims = user_sim[user_idx]
    pred = np.zeros(R.shape[1])
    for i in range(R.shape[1]):
        if R[user_idx, i] > 0: pred[i] = -1; continue
        raters = np.where(R[:, i] > 0)[0]
        if len(raters) > 0 and sims[raters].sum() > 0:
            pred[i] = np.dot(sims[raters], R[raters, i]) / sims[raters].sum()
    top_idx = np.argsort(pred)[::-1][:top_k]
    return [(item_names[i], round(pred[i], 3)) for i in top_idx if pred[i] > 0]

In [ ]:
#R = 사용자-아이템 평점 행렬 (rating matrix)

R = np.zeros((len(user_types), len(etf_data)))
for u, ut in enumerate(user_types):
    for i, etf in enumerate(etf_data):
        r = seed_rating(ut, etf)
        if r > 0 and np.random.random() > 0.3:
            R[u, i] = r

# 예시)
#            KODEX200  채권ETF  리츠ETF  ...
# U0(보수)      0        5       4
# U1(보수)      2        5       0     ← 0 = 미평가
# U4(공격)      3        0       0

---
### 실습 6: 프로필 기반 추천 + 포트폴리오 리스크 분석

자연어 메시지에서 프로필을 추출하고, LLM 추천 + 리스크 검증을 실행하세요.

**구현할 것**:
1. `analyze_risk(profile, recs, etf_data)` — 카테고리 집중도/위험 불일치/수수료 과다 경고
2. `explain_risks(profile, warnings)` — LLM으로 100자 이내 설명
3. 2개 이상의 프로필로 테스트하고 결과 비교

In [165]:
  etf_dict = {e["name"]: e for e in etf_data}
  warnings = []
  rec_etfs = [etf_dict[r["name"]] for r in recommendations if r["name"] in etf_dict]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

NameError: name 'recommendations' is not defined

In [166]:
# 실습 6
def analyze_risk(profile, recommendations, etf_data):
    """리스크 경고 생성"""
    etf_dict = {e["name"]: e for e in etf_data}
    warnings = []
    rec_etfs = [etf_dict[r["name"]] for r in recommendations if r["name"] in etf_dict]
    if not rec_etfs:
        return warnings
    # ---- 여기에 코드 작성 ----
    # 1) 카테고리 집중도 > 60%
    cat_alloc = {}
    total = 0
    for r in recommendations:
        if r["name"] not in etf_dict:
            continue
        cat = etf_dict[r["name"]]["category"]
        alloc = r.get("allocation", 100 / len(recommendations))
        cat_alloc[cat] = cat_alloc.get(cat, 0) + alloc
        total += alloc
    for cat, alloc in cat_alloc.items():
        pct = alloc / total * 100 if total else 0
        if pct > 60:
            warnings.append(
                f"카테고리 집중도 경고: '{cat}' 비중이 {pct:.0f}% (60% 초과)"
            )
    # 2) 위험 불일치
    risk_rank = {"낮음": 1, "중간": 2, "높음": 3}
    user_risk = risk_rank.get(profile.risk_tolerance, 2)
    for etf in rec_etfs:
        etf_risk = risk_rank.get(etf["risk_level"], 2)
        if etf_risk > user_risk:
            warnings.append(
                f"위험 불일치: {etf['name']}({etf['risk_level']}) > 사용자 성향({profile.risk_tolerance})"
            )
    # 3) 수수료 > 0.5%
    for etf in rec_etfs:
        if etf["expense_ratio"] > 0.5:
            warnings.append(
                f"수수료 과다: {etf['name']} 운용보수 {etf['expense_ratio']}% (0.5% 초과)"
            )

    return warnings

def explain_risks(profile, warnings):
    """LLM으로 리스크 설명"""
    if not warnings:
        return "특별한 리스크 경고가 없습니다."

    prompt = f"""투자 프로필과 리스크 경고 목록이 주어졌을 때, 투자자가 주의해야 할 핵심을 100자 이내로 설명하세요.

    [프로필]
    - 이름: {profile.name}
    - 위험 성향: {profile.risk_tolerance}
    - 투자 목표: {profile.investment_goal}

    [리스크 경고]
    {chr(10).join(f'- {w}' for w in warnings)}

    조건: 100자 이내, 한 문단, 경고의 핵심만."""

    return llm.invoke(prompt).content.strip()

# 2개 프로필 테스트
test_users = [
    "30대 공격적 투자자, 월 200만원, 자산증식 목표",
    "55세 은퇴 준비, 월 100만원, 안정적 배당 중시",
]
for msg in test_users:
    profile = extract_profile(msg)
    cands = rule_based_filter(profile.risk_tolerance, etf_data)
    recs = llm_recommend(profile, cands, top_k=3)
    ws = analyze_risk(profile, recs, etf_data)
    print(f"\n[{profile.name}] {profile.risk_tolerance}/{profile.investment_goal}")
    for r in recs:
        print(f"  {r.get('allocation', '?')}% | {r['name']}")
    print(f"  리스크: {explain_risks(profile, ws)}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


[사용자] 중립/자산증식
  50% | KODEX 미국S&P500TR
  30% | ACE 미국배당다우존스
  20% | KODEX 200
  리스크: 특별한 리스크 경고가 없습니다.

[사용자] 중립/자산증식
  50% | KODEX 미국S&P500TR
  30% | ACE 미국배당다우존스
  20% | KODEX 200
  리스크: 특별한 리스크 경고가 없습니다.


#### 기존 코드

In [ ]:
# def llm_recommend(profile, candidates, top_k=3):
#     """LLM 추천 + allocation + reason"""
#     etf_list = "\n".join([ ...

---
### 실습 7 (최종): 통합 파이프라인 + LLM 리포트

모든 부품을 연결하여 **원클릭 추천 파이프라인**을 완성하고, LLM으로 최종 마크다운 리포트를 생성하세요.

**파이프라인**:
```
자연어 → 프로필 추출 → 규칙 필터 → LLM 추천
      → 리스크 분석 → LLM 설명 → 마크다운 리포트
```

**구현할 것**:
1. `recommend_pipeline(user_text)` — 전체 파이프라인 실행
2. `final_report(result)` — LLM 마크다운 리포트 생성 (투자자 요약 / 추천 / 리스크 / 다음 단계)

In [167]:
# 실습 7
def recommend_pipeline(user_text, top_k=3):
    """End-to-End 추천 파이프라인"""
    # ---- 여기에 코드 작성 ----
    # 1) extract_profile
    profile = extract_profile(user_text)

    # 2) rule_based_filter
    candidates = rule_based_filter(profile.risk_tolerance, etf_data)

    # 3) llm_recommend
    recommendations = llm_recommend(profile, candidates, top_k=top_k)

    # 4) analyze_risk + explain_risks
    warnings = analyze_risk(profile, recommendations, etf_data)
    risk_explanation = explain_risks(profile, warnings)

    # 5) 결과 dict 반환
    return {
        "user_text": user_text,
        "profile": profile,
        "candidates_count": len(candidates),
        "recommendations": recommendations,
        "warnings": warnings,
        "risk_explanation": risk_explanation,
    }

def final_report(pipeline_result):
    """LLM 마크다운 리포트"""
    # ---- 여기에 코드 작성 ----
    profile = pipeline_result["profile"]
    recs = pipeline_result["recommendations"]
    warnings = pipeline_result["warnings"]

    recs_text = "\n".join(
        f"- **{r['name']}** ({r.get('allocation', '?')}%): {r.get('reason', '')}"
        for r in recs
    )
    warnings_text = "\n".join(f"- {w}" for w in warnings) if warnings else "- 특이사항 없음"

    prompt = f"""당신은 ETF 포트폴리오 리포트를 작성하는 투자 자문가입니다.
    아래 자료를 바탕으로 **마크다운** 형식의 투자 리포트를 작성하세요.

    ## 입력 자료
    - 사용자 요청: "{pipeline_result['user_text']}"
    - 프로필: {profile.name} / {profile.risk_tolerance} / {profile.investment_goal} / {profile.investment_horizon}년 / 월 {profile.monthly_budget}만원
    - 추천 포트폴리오:
    {recs_text}
    - 리스크 경고:
    {warnings_text}

    ## 리포트 작성 규칙
    다음 4개 섹션을 **정확히 이 순서**로 포함하세요:

    # 투자자 요약
    (프로필을 한 문단으로, 3~4줄)

    # 추천 포트폴리오
    (각 ETF를 표 또는 불릿으로, 비중과 추천 이유 포함)

    # 리스크 분석
    (위 리스크 경고를 투자자 눈높이로 풀어서 설명, 없으면 '균형잡힌 포트폴리오' 같이 긍정 멘트)

    # 다음 단계
    (실행 체크리스트 3~4개 — 계좌개설·분할매수·리밸런싱 주기 등)

    **중요**: 순수 마크다운만 출력하세요. 코드블록 감싸기 금지."""

    return llm.invoke(prompt).content


# 실행
result = recommend_pipeline("은퇴 준비 중인 55세입니다. 월 200만원 여유가 있고 안정적인 배당이 중요해요.")
print(final_report(result))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 투자자 요약
55세의 은퇴 준비 중인 투자자로, 월 200만원의 여유 자금을 가지고 있으며 안정적인 배당 수익을 중요시합니다. 중립적인 투자 성향을 가지고 있으며, 자산 증식을 목표로 하고 있습니다. 5년의 투자 기간을 설정하고 월 100만원을 투자할 계획입니다.

# 추천 포트폴리오
| ETF 이름                     | 비중  | 추천 이유                                                   |
|------------------------------|-------|------------------------------------------------------------|
| KODEX 미국S&P500TR          | 50%   | 높은 수익률(25.3%)과 낮은 수수료(0.05%)로 장기적인 자산 증식에 유리. |
| ACE 미국배당다우존스        | 30%   | 안정적인 배당 수익(3.5%)과 낮은 위험으로 안정성을 제공.         |
| KODEX 200                   | 20%   | 국내 주식에 대한 투자로 포트폴리오 다각화 및 중간 수익률(8.5%)을 기대. |

# 리스크 분석
현재 추천된 포트폴리오는 균형 잡힌 구성을 가지고 있어 안정성과 수익성을 동시에 추구할 수 있습니다. 각 ETF의 특성과 비중이 잘 조화되어 있어, 시장 변동성에 대한 리스크를 최소화할 수 있습니다.

# 다음 단계
1. 투자 계좌 개설: 증권사에 계좌를 개설하여 ETF 거래를 시작합니다.
2. 분할 매수: 월 100만원씩 정기적으로 분할 매수하여 시장 타이밍의 리스크를 줄입니다.
3. 리밸런싱 주기 설정: 6개월마다 포트폴리오 비중을 점검하고 필요시 리밸런싱을 실시합니다.
4. 배당 수익 재투자: 배당금을 재투자하여 복리 효과를 극대화합니다.


---
### Weekend 3 완료!

| 실습 | 핵심 |
|------|------|
| 1 | split vs Kiwi **정량 비교** — 토크나이저 품질이 BM25 성능 좌우 |
| 2 | **쿼리 의도 분류** + 검색 전략 자동 라우팅 |
| 3 | **α 최적화** + FAISS/BM25/Hybrid 비교 |
| 4 | **MAP@K + LLM-as-Judge** — 새로운 검색 평가 메트릭 |
| 5 | CBF **다양성** + cold-start **폴백** |
| 6 | LLM 추천 + **포트폴리오 리스크** 분석 |
| 7 | **통합 파이프라인** + LLM 마크다운 리포트 |